# GAT 从零实现：账户关系图中的风险识别

## 面试问题

面试时我会先说明：GAT 的核心不是给邻居做普通平均，而是对每条入边计算可学习的相关性，再在同一目标节点的入边内做 softmax。节点特征先经过共享线性变换，源节点与目标节点共同决定注意力分数。聚合必须按目标节点归一化，否则不同度数节点的数值尺度不可比。自环很重要，因为它让节点在聚合邻居时保留自己的证据，也让孤立节点仍有表示。工程上还要考虑边类型、时间衰减、超大图采样和新节点冷启动。下面用账户风险图比较只看账户特征的基线与手写 edge attention，并检查真实注意力和孤立节点失败案例。

## 真实案例

数据是 10 个脱敏电商账户的离线快照：特征依次是账户年龄、拒付率、设备风险，边表示近期共享设备或收货地址。标签由历史人工调查给出；这里简化了时间、边类型和标签延迟。A005、A009 的单点特征故意较模糊，但其关系邻居集中在风险社区。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import torch  # 导入 PyTorch 以实现可求导的图注意力网络。
from torch import nn  # 导入神经网络基础模块以声明参数层。
import torch.nn.functional as F  # 导入激活函数和损失函数等基础算子。
torch.manual_seed(27)  # 固定随机种子以保证教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验的运行抖动。
account_ids = ["A001", "A002", "A003", "A004", "A005", "A006", "A007", "A008", "A009", "A010"]  # 定义十个可读账户编号。
features = torch.tensor([[0.10, 0.82, 0.90], [0.15, 0.76, 0.84], [0.78, 0.12, 0.10], [0.70, 0.16, 0.14], [0.42, 0.46, 0.44], [0.45, 0.48, 0.46], [0.52, 0.32, 0.28], [0.28, 0.55, 0.52], [0.40, 0.47, 0.43], [0.38, 0.50, 0.48]], dtype=torch.float32)  # 保存账户年龄、拒付率和设备风险三个特征。
labels = torch.tensor([1, 1, 0, 0, 1, 0, 0, 1, 1, 0], dtype=torch.long)  # 保存人工调查得到的风险标签。
undirected_edges = [(0, 1), (0, 4), (1, 4), (1, 7), (4, 7), (4, 8), (7, 8), (2, 3), (2, 5), (3, 5), (3, 6), (5, 6)]  # 定义共享设备或地址形成的社区边。
directed_edges = undirected_edges + [(target, source) for source, target in undirected_edges]  # 把无向关系展开为双向消息边。
graph_edges = directed_edges + [(index, index) for index in range(len(account_ids))]  # 加入自环以保留节点自身证据。
edge_index = torch.tensor(graph_edges, dtype=torch.long).t().contiguous()  # 转为源节点行和目标节点行组成的边索引。
print("账户  年龄  拒付率  设备风险  标签")  # 打印输入表头方便核对业务字段。
for index, account_id in enumerate(account_ids):  # 逐账户展示原始输入而不是只展示张量形状。
    print(f"{account_id}  {features[index, 0]:.2f}  {features[index, 1]:.2f}    {features[index, 2]:.2f}      {'风险' if labels[index] else '正常'}")  # 输出当前账户的特征和监督标签。
print(f"图结构：节点数={len(account_ids)}，有向边含自环={edge_index.shape[1]}，特征形状={tuple(features.shape)}")  # 输出图规模与张量形状。

账户  年龄  拒付率  设备风险  标签
A001  0.10  0.82    0.90      风险
A002  0.15  0.76    0.84      风险
A003  0.78  0.12    0.10      正常
A004  0.70  0.16    0.14      正常
A005  0.42  0.46    0.44      风险
A006  0.45  0.48    0.46      正常
A007  0.52  0.32    0.28      正常
A008  0.28  0.55    0.52      风险
A009  0.40  0.47    0.43      风险
A010  0.38  0.50    0.48      正常
图结构：节点数=10，有向边含自环=34，特征形状=(10, 3)


## 基线：只看单账户阈值

最便宜的规则把拒付率与设备风险相加后设阈值。它能识别证据很强的账户，却看不到 A005、A009 周围的风险社区；基线与 GAT 使用完全相同的十个账户和准确率指标。

In [2]:
baseline_scores = features[:, 1] + features[:, 2]  # 把拒付率和设备风险相加形成单点风险分数。
baseline_predictions = (baseline_scores > 1.0).long()  # 用固定阈值得到不使用图关系的预测。
baseline_accuracy = (baseline_predictions == labels).float().mean().item()  # 在同一批账户上计算基线准确率。
print("账户  基线分数  基线预测  真实标签")  # 打印逐账户对照表的表头。
for index, account_id in enumerate(account_ids):  # 遍历所有账户观察规则具体错在哪里。
    print(f"{account_id}  {baseline_scores[index]:.2f}      {baseline_predictions[index].item()}         {labels[index].item()}")  # 输出每个账户的基线分数和标签。
print(f"基线准确率={baseline_accuracy:.1%}")  # 汇总不使用图结构的基线表现。

账户  基线分数  基线预测  真实标签
A001  1.72      1         1
A002  1.60      1         1
A003  0.22      0         0
A004  0.30      0         0
A005  0.90      0         1
A006  0.94      0         0
A007  0.60      0         0
A008  1.07      1         1
A009  0.90      0         1
A010  0.98      0         0
基线准确率=80.0%


## 手写核心：按目标节点归一化的 edge attention

`EdgeGATLayer` 明确完成四步：共享线性变换、逐边打分、同一目标节点入边 softmax、按权重聚合。这里没有导入 PyG/DGL 的现成 GAT；`index_add_` 只负责把已经算好的消息累加到目标节点。

In [3]:
class EdgeGATLayer(nn.Module):  # 定义从边级注意力开始实现的单头 GAT 层。
    def __init__(self, input_dim, output_dim):  # 接收输入维度和输出维度以创建可学习参数。
        super().__init__()  # 初始化父类以正确注册所有参数。
        self.weight = nn.Parameter(torch.randn(input_dim, output_dim) * 0.25)  # 创建共享节点线性变换矩阵。
        self.source_attention = nn.Parameter(torch.randn(output_dim) * 0.20)  # 创建源节点注意力向量。
        self.target_attention = nn.Parameter(torch.randn(output_dim) * 0.20)  # 创建目标节点注意力向量。
    def forward(self, node_features, edges):  # 根据节点特征和有向边计算新表示与边权重。
        source_nodes = edges[0]  # 读取每条消息边的源节点编号。
        target_nodes = edges[1]  # 读取每条消息边的目标节点编号。
        transformed = node_features @ self.weight  # 对所有节点应用同一线性变换。
        source_scores = (transformed[source_nodes] * self.source_attention).sum(dim=1)  # 计算每条边的源节点贡献。
        target_scores = (transformed[target_nodes] * self.target_attention).sum(dim=1)  # 计算每条边的目标节点贡献。
        edge_scores = F.leaky_relu(source_scores + target_scores, negative_slope=0.2)  # 合并两端证据并应用 LeakyReLU。
        attention = torch.zeros_like(edge_scores)  # 创建与边数一致的注意力容器。
        for target in range(node_features.shape[0]):  # 逐目标节点独立归一化其所有入边。
            incoming_mask = target_nodes == target  # 找出当前目标节点对应的全部入边。
            attention[incoming_mask] = torch.softmax(edge_scores[incoming_mask], dim=0)  # 在当前入边集合内执行 softmax。
        messages = attention.unsqueeze(1) * transformed[source_nodes]  # 用注意力权重缩放每条源节点消息。
        aggregated = torch.zeros_like(transformed)  # 创建目标节点的聚合结果张量。
        aggregated.index_add_(0, target_nodes, messages)  # 按目标节点累加所有加权消息。
        return aggregated, attention  # 返回节点新表示和可解释的逐边注意力。
class AccountRiskGAT(nn.Module):  # 定义用于账户风险二分类的小型 GAT。
    def __init__(self):  # 创建图注意力层和分类头。
        super().__init__()  # 初始化父类以注册子模块。
        self.gat = EdgeGATLayer(3, 8)  # 把三个账户特征聚合成八维图表示。
        self.classifier = nn.Linear(8, 2)  # 把图表示映射为正常与风险两个 logits。
    def forward(self, node_features, edges):  # 完成一次图消息传递和分类前向计算。
        aggregated, attention = self.gat(node_features, edges)  # 计算注意力加权的邻居表示。
        hidden = F.elu(aggregated)  # 用 ELU 引入非线性以表达社区模式。
        logits = self.classifier(hidden)  # 输出每个账户的二分类 logits。
        return logits, attention, hidden  # 返回预测、中间注意力和隐藏表示。
model = AccountRiskGAT()  # 实例化手写账户风险 GAT。
parameter_count = sum(parameter.numel() for parameter in model.parameters())  # 统计模型可训练参数规模。
print(model)  # 展示手写网络的实际层次结构。
print(f"可训练参数量={parameter_count}")  # 输出参数量以明确教学模型规模。

AccountRiskGAT(
  (gat): EdgeGATLayer()
  (classifier): Linear(in_features=8, out_features=2, bias=True)
)
可训练参数量=58


In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)  # 创建 Adam 优化器更新手写 GAT 参数。
loss_trace = []  # 保存训练损失以观察真实优化过程。
first_gradient_norm = 0.0  # 预留首轮梯度范数用于证明发生了反向传播。
for epoch in range(301):  # 在受控小图上执行三百零一次全图训练。
    optimizer.zero_grad()  # 清空上一轮残留梯度。
    logits, attention, hidden = model(features, edge_index)  # 运行手写 edge attention 前向传播。
    loss = F.cross_entropy(logits, labels)  # 用人工调查标签计算交叉熵损失。
    loss.backward()  # 反向传播到线性变换与注意力向量。
    if epoch == 0:  # 只在首轮记录一个代表性梯度规模。
        first_gradient_norm = model.gat.weight.grad.norm().item()  # 读取图变换矩阵的真实梯度范数。
    optimizer.step()  # 根据当前梯度更新全部模型参数。
    loss_trace.append(loss.item())  # 保存当前轮损失用于前后对比。
    if epoch in [0, 50, 150, 300]:  # 选择少量关键轮次输出训练轨迹。
        current_accuracy = (logits.argmax(dim=1) == labels).float().mean().item()  # 计算当前全图分类准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} accuracy={current_accuracy:.1%}")  # 输出损失和准确率的真实变化。
model.eval()  # 切换到评估模式以生成稳定结果。
with torch.no_grad():  # 关闭评估阶段的梯度记录以节省内存。
    gat_logits, gat_attention, gat_hidden = model(features, edge_index)  # 重新计算最终 logits、注意力和隐藏表示。
gat_predictions = gat_logits.argmax(dim=1)  # 选择每个账户概率最大的类别。
gat_probabilities = torch.softmax(gat_logits, dim=1)[:, 1]  # 提取风险类别概率供逐样本解释。
gat_accuracy = (gat_predictions == labels).float().mean().item()  # 计算 GAT 在同一批账户上的准确率。
target_index = account_ids.index("A005")  # 定位特征模糊但处于风险社区的 A005。
incoming_positions = torch.where(edge_index[1] == target_index)[0]  # 找出所有指向 A005 的边位置。
print(f"首轮图权重梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明模型确实被训练。
print("A005 的入边注意力：")  # 标记即将展示的可解释边权重。
for position in incoming_positions.tolist():  # 逐条展示进入 A005 的消息来源和权重。
    source_index = edge_index[0, position].item()  # 读取当前入边的源节点编号。
    print(f"  {account_ids[source_index]} -> A005: {gat_attention[position].item():.4f}")  # 输出源账户与归一化注意力。

epoch=000 loss=0.6979 accuracy=50.0%


epoch=050 loss=0.0634 accuracy=100.0%


epoch=150 loss=0.0074 accuracy=100.0%


epoch=300 loss=0.0016 accuracy=100.0%
首轮图权重梯度范数=0.079615
A005 的入边注意力：
  A001 -> A005: 0.9714
  A002 -> A005: 0.0286
  A008 -> A005: 0.0000
  A009 -> A005: 0.0000
  A005 -> A005: 0.0000


## 结果解读

下面逐账户比较同一数据上的规则基线与 GAT。重点不是小数据上的绝对准确率，而是 A005、A009 这类“单点证据不够、邻接关系有信息”的账户是否被修正。A005 的入边权重之和应为 1，这正是按目标节点归一化的含义。

In [5]:
print("账户  真实  基线  GAT风险概率  GAT预测")  # 打印逐账户实验结果表头。
for index, account_id in enumerate(account_ids):  # 遍历十个账户输出可读预测结果。
    print(f"{account_id}   {labels[index].item()}     {baseline_predictions[index].item()}      {gat_probabilities[index]:.3f}       {gat_predictions[index].item()}")  # 对照真实标签、基线和 GAT。
print(f"同数据准确率：基线={baseline_accuracy:.1%}，GAT={gat_accuracy:.1%}")  # 汇总两个方案的同口径指标。

账户  真实  基线  GAT风险概率  GAT预测
A001   1     1      1.000       1
A002   1     1      1.000       1
A003   0     0      0.000       0
A004   0     0      0.000       0
A005   1     0      1.000       1
A006   0     0      0.000       0
A007   0     0      0.000       0
A008   1     1      1.000       1
A009   1     0      0.992       1
A010   0     0      0.008       0
同数据准确率：基线=80.0%，GAT=100.0%


## 失败案例：删除自环后孤立账户丢失自身证据

A010 没有共享设备边。若构图时漏掉自环，它没有任何入边，聚合向量会变成全零；加入自环后至少能把自己的账户特征送入分类器。这不是调高阈值能修复的问题，而是消息传递图本身缺边。

In [6]:
edges_without_self = torch.tensor(directed_edges, dtype=torch.long).t().contiguous()  # 构造故意遗漏自环的错误图。
with torch.no_grad():  # 关闭梯度以单独观察构图差异。
    hidden_without_self, attention_without_self = model.gat(features, edges_without_self)  # 在错误图上执行同一注意力层。
    hidden_with_self, attention_with_self = model.gat(features, edge_index)  # 在含自环的正确图上执行同一层。
isolated_index = account_ids.index("A010")  # 定位没有共享关系边的孤立账户。
isolated_bad_norm = hidden_without_self[isolated_index].norm().item()  # 计算错误构图下孤立节点表示范数。
isolated_fixed_norm = hidden_with_self[isolated_index].norm().item()  # 计算加入自环后的节点表示范数。
print(f"A010 无自环表示范数={isolated_bad_norm:.6f}")  # 展示错误构图导致的信息归零。
print(f"A010 加自环表示范数={isolated_fixed_norm:.6f}")  # 展示修复后自身证据被保留下来。
print("修复结论：孤立节点必须有自环或单独的自身特征旁路。")  # 给出面向工程实现的明确结论。

A010 无自环表示范数=0.000000
A010 加自环表示范数=1.291555
修复结论：孤立节点必须有自环或单独的自身特征旁路。


## 生产差距

线上图会有亿级节点、异构边、时间漂移和调查标签延迟，需要邻居采样、关系类型编码、时间衰减、分布式图存储与离线/在线特征一致性。注意力权重只能解释模型聚合了谁，不能直接当作因果证据。还要监控孤立点比例、度数分布、可疑团伙召回率以及误杀成本。

## 最小回归测试

In [7]:
assert len(account_ids) >= 6  # 保证案例包含足够多的真实语义节点。
assert gat_accuracy > baseline_accuracy  # 保证图关系在本受控案例中修正了基线错误。
assert gat_accuracy == 1.0  # 保证手写 GAT 已拟合这张教学小图。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使训练损失下降。
assert first_gradient_norm > 0.0  # 保证首轮图权重获得了非零梯度。
assert torch.isclose(gat_attention[incoming_positions].sum(), torch.tensor(1.0), atol=1e-5)  # 保证 A005 入边注意力按目标节点归一化。
assert isolated_bad_norm == 0.0 and isolated_fixed_norm > 0.0  # 保证自环确实修复孤立节点信息归零。
print("回归测试通过：注意力归一化、训练更新和自环修复均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：注意力归一化、训练更新和自环修复均符合预期。
